In [2]:
!pip install pandas scikit-learn openpyxl

In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import pickle
import warnings
warnings.filterwarnings("ignore")

# Preprocessor function to convert to lowercase, remove extra spaces and punctuation.
def text_preprocess(text):
    text = str(text).lower().strip()            
    text = re.sub(r'\s+', ' ', text)             
    text = re.sub(r'[^\w\s]', '', text)    
    return text


In [4]:
# read and load data dataset
excel_file = "../data/categories.xlsx"

df = pd.read_excel(excel_file)

df.head()


,Types,Personal data,Credentials for others resources,HR data,Financial documents,Documents marked as confidential
0,Keywords,surname,username,accounts,invoice,confidential
1,NaN,passport,password,human resources,check,classified
2,NaN,tax identification,credential,hr,receipt,private
3,NaN,insurance number,login,employee,payment,restricted
4,NaN,phone,authentication,staff,payment confirmation,contract


In [5]:
# Create a list to save samples and labels
data = []
labels = []

for col in df.columns:
    keywords = df[col].dropna().astype(str).tolist()
    for keyword in keywords:
        data.append(text_preprocess(keyword))
        labels.append(col)

train_df = pd.DataFrame({'text': data, 'label': labels})
train_df.head()


,text,label
0,keywords,Types
1,surname,Personal data
2,passport,Personal data
3,tax identification,Personal data
4,insurance number,Personal data


In [6]:
# Split the dataset into two parts: training set and test set.
print("Number of samples per layer before filtering:")
print(train_df['label'].value_counts())

min_samples_required = 2
valid_classes = train_df['label'].value_counts()[train_df['label'].value_counts() >= min_samples_required].index

filtered_df = train_df[train_df['label'].isin(valid_classes)]

X = filtered_df['text']
y = filtered_df['label']

print("Number of samples for each layer after filtering:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Number of training samples:", len(X_train))
print("Number of testing samples:", len(X_test))


Number of samples per layer before filtering:
label
Credentials for others resources    17
HR data                             17
Financial documents                 17
Personal data                       13
Documents marked as confidential    12
Types                                1
Name: count, dtype: int64
Number of samples for each layer after filtering:
label
Credentials for others resources    17
HR data                             17
Financial documents                 17
Personal data                       13
Documents marked as confidential    12
Name: count, dtype: int64
Number of training samples: 60
Number of testing samples: 16


In [7]:
# Convert text into numeric vectors for processing.
vectorizer = TfidfVectorizer(ngram_range=(1,2))

X_train_vect = vectorizer.fit_transform(X_train)

X_test_vect = vectorizer.transform(X_test)


In [8]:
# Train a Classification Model with Logistic Regression
clf = LogisticRegression(solver="liblinear", random_state=42)

clf.fit(X_train_vect, y_train)


LogisticRegression(random_state=42, solver='liblinear')

In [9]:
# Evaluate the performance of the model after training
y_pred = clf.predict(X_test_vect)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))


Accuracy: 0.25

Classification report:
                                   precision    recall  f1-score   support

Credentials for others resources       0.20      1.00      0.33         3
Documents marked as confidential       0.00      0.00      0.00         2
             Financial documents       0.00      0.00      0.00         4
                         HR data       1.00      0.25      0.40         4
                   Personal data       0.00      0.00      0.00         3

                        accuracy                           0.25        16
                       macro avg       0.24      0.25      0.15        16
                    weighted avg       0.29      0.25      0.16        16



In [10]:
# Save trained model and vectorizer
with open("model_trained.pkl", "wb") as f:
    pickle.dump(clf, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model saved (model_trained.pkl) and vectorizer (vectorizer.pkl).")


Model saved (model_trained.pkl) and vectorizer (vectorizer.pkl).
